# Chapter 16 &mdash; Variable Ordering, and What BDDs Do Not Settle

**Concept 18 of the Chapter 16 decomposition:** *Variable Ordering, and What BDDs Do Not Settle*

The same function under two orders can differ exponentially; finding the best order is itself NP-complete. The cost was moved, not removed.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter16-NPC/Concept-Variable-Ordering-And-P-Versus-NP/Concept-Variable-Ordering-And-P-Versus-NP.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Bdd            import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


A BDD is built against a **fixed variable order**, and the order is not a detail.

For $(a_1\wedge b_1)\vee\cdots\vee(a_n\wedge b_n)$, interleaving the pairs gives
$2n+2$ nodes; grouping all the $a$'s before all the $b$'s gives $2^{n+1}$. Linear
against exponential, for the *same Boolean function*.

Worse, you cannot reliably guess which way it will go. On the colouring map of
Concept 17 the **grouped** order is the smaller one &mdash; the reverse of the pairing
intuition that the family above teaches.

And this is not a failure of effort. Deciding whether a variable ordering of a given
size exists is **itself NP-complete**, so hunting for a good order is a problem in the
same family as the one you were trying to solve.

Which is the honest ending. A BDD decides SAT without searching (Concept 13), reads off
both normal forms (14), stores an exponential DNF compactly (15), counts models for free
(16), and solves a colouring outright (17). **None of that puts SAT in P**, because
every one of those operations is linear in a structure that can be exponentially large.

The cost was moved, not removed. That is worth knowing precisely, because moving a cost
is often exactly what makes a problem tractable *in practice* &mdash; and never what
makes it tractable *in theory*.

## 2. Definitions

### One function, two orders

In [ ]:
def pairs(n, interleave):
    order = (' '.join('a%d b%d' % (i, i) for i in range(1, n + 1))
             if interleave else
             ' '.join('a%d' % i for i in range(1, n + 1)) + ' ' +
             ' '.join('b%d' % i for i in range(1, n + 1)))
    body = ' | '.join('(a%d & b%d)' % (i, i) for i in range(1, n + 1))
    return 'Var_Order : %s\nf = %s\nMain_Exp : f' % (order, body)

print(pairs(3, True))

<!-- nav-strip -->

---

&larr;&nbsp;[Ch16&nbsp;17.&nbsp;Graph Colouring as a Boolean Formula](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter16-NPC/Concept-Graph-Colouring-As-A-Formula/Concept-Graph-Colouring-As-A-Formula.ipynb) &nbsp;&middot;&nbsp; [**Chapter 16** index](https://github.com/ganeshutah/Jove/blob/master/Chapter16-NPC/README.md) &nbsp;&middot;&nbsp; [Ch17&nbsp;1.&nbsp;Boolean Functions, Truth-Table Personalities, and Why Tables Do Not Scale](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter17-BDD/Concept-Truth-Tables-Do-Not-Scale/Concept-Truth-Tables-Do-Not-Scale.ipynb)&nbsp;&rarr;

---

## 3. Tests

Same function each time. Only the order differs.

In [ ]:
print('%3s %6s %14s %14s' % ('n', 'vars', 'interleaved', 'grouped'))
for n in range(2, 8):
    g = bdd(pairs(n, True))
    b = bdd(pairs(n, False))
    assert g.count == b.count          # the same function, either way
    print('%3s %6d %14d %14d' % (n, 2 * n, g.nodes, b.nodes))
print()
print('interleaved = 2n+2      grouped = 2^(n+1)')
print('Linear against exponential, for one Boolean function.')

**And now look at them.** Same Boolean function, drawn twice. The order is the only thing that differs.

In [ ]:
side_by_side(('interleaved  a1 b1 a2 b2 a3 b3  -- %d nodes'
              % bdd(pairs(3, True)).nodes,  bdd(pairs(3, True))),
             ('grouped  a1 a2 a3 b1 b2 b3  -- %d nodes'
              % bdd(pairs(3, False)).nodes, bdd(pairs(3, False))))

**Why interleaving wins here.** Reading $a_i$ then $b_i$ lets the diagram *forget* the pair as soon as it is resolved. Grouping forces it to remember every $a$ until the matching $b$ arrives &mdash; and remembering $n$ bits takes $2^n$ nodes.

In [ ]:
for n in (3, 6):
    print('n=%d: grouped has %d nodes; 2^(n+1) = %d'
          % (n, bdd(pairs(n, False)).nodes, 2 ** (n + 1)))
print()
print('The diagram has to encode WHAT IT HAS SEEN SO FAR.  That is the')
print('same Myhill-Nerode argument as Chapter 17: a BDD is the minimal DFA')
print('of the function, and these are its distinguishable states.')

**But do not trust the intuition.** On the colouring map from Concept 17, interleaving is the *worse* choice.

In [ ]:
FOURCOL = '''
Var_Order : aUT bUT aNV bNV aAZ bAZ aCO bCO

Nv_not_Ut = ~((aNV <=> aUT) & (bNV <=> bUT))
Nv_not_Az = ~((aNV <=> aAZ) & (bNV <=> bAZ))
Az_not_Ut = ~((aUT <=> aAZ) & (bUT <=> bAZ))
Co_not_Az = ~((aCO <=> aAZ) & (bCO <=> bAZ))
Co_not_Ut = ~((aCO <=> aUT) & (bCO <=> bUT))

Main_Exp : Nv_not_Ut & Nv_not_Az & Az_not_Ut & Co_not_Az & Co_not_Ut
'''

def colour_of(model, state):
    # the two bits of a state, read as a number 0..3
    return 2 * model['a' + state] + model['b' + state]

GROUPED = FOURCOL.replace('Var_Order : aUT bUT aNV bNV aAZ bAZ aCO bCO',
                          'Var_Order : aUT aNV aAZ aCO bUT bNV bAZ bCO')

inter = bdd(FOURCOL)
group = bdd(GROUPED)
print('interleaved (a,b per state) : %d nodes' % inter.nodes)
print('grouped (all a, then all b) : %d nodes' % group.nodes)
print('same function?', inter.count == group.count, '(%d colourings each)'
      % inter.count)
assert inter.count == group.count
print()
print('The reverse of the previous result.  Which order wins depends on the')
print('function, and deciding whether an ordering below a given size EXISTS')
print('is itself NP-complete -- so this is not a gap you can close by')
print('thinking harder about it.')

### The accounting\n\nWhat these six concepts did and did not show.

In [ ]:
rows = [('decide SAT without searching', 'Concept 13',
         'linear in the diagram'),
        ('read off DNF and CNF',         'Concept 14',
         'linear in the number of paths'),
        ('hold an exponential DNF',      'Concept 15',
         'linear nodes, 4^k paths'),
        ('count all models (#SAT)',      'Concept 16',
         'linear in the diagram'),
        ('solve a graph colouring',      'Concept 17',
         'linear in the diagram'),
        ('choose the variable order',    'Concept 18',
         'NP-complete')]
W = [max(len(r[i]) for r in rows) for i in range(3)]
hdr = ('what a BDD does', 'where', 'what it costs')
W = [max(w, len(h)) for w, h in zip(W, hdr)]
line = lambda cs, p=' ': '|' + '|'.join(p + c.ljust(w, p) + p
                                        for c, w in zip(cs, W)) + '|'
print(line(hdr))
print(line(('', '', ''), '-'))
for r in rows:
    print(line(r))
print()
print('Every "linear" above is linear IN THE DIAGRAM, and the diagram is')
print('what can be exponential.  SAT is exactly where Cook and Levin left')
print('it.  What changed is that the exponential is now somewhere you can')
print('see it, measure it, and sometimes -- with a good order -- avoid it.')

## 4. Exercises


1. For `pairs(n, False)` the node count is $2^{n+1}$. Prove it by identifying the
   distinguishable prefixes, in the Myhill&ndash;Nerode sense of Chapter 17.
2. Find a variable order for the colouring map that beats **both** of the two tried
   here, or convince yourself none exists. How much of the search space did you cover?
3. "Deciding whether an ordering of size $\le k$ exists is NP-complete." Does that
   make BDD construction useless in practice? Chapter 17 discusses dynamic reordering;
   say what problem it actually solves.
4. Concept 15 showed linear nodes with exponentially many paths. Concept 18 shows
   exponential nodes. Give a formula family exhibiting each, and say which is the
   worse position to be in.
5. Suppose someone reports a BDD package that keeps every diagram polynomial in the
   formula size. What would follow? Name the collapse precisely.

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for all 255 concepts.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter16-NPC/Concept-Variable-Ordering-And-P-Versus-NP')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')